# galrax quickstart

Certified zero sets of integer orbits (the Skolem problem and its nonlinear version), exact $p$-adic flows and jump-ahead for maps of $\mathbb Z/2^{64}$. Every answer below is exact; a `ZeroSet` with `certified=True` comes with a certificate that its list of zeros is complete.

In [1]:
%pip install -q 'galrax[abs]'

In [2]:
import jax


jax.config.update("jax_enable_x64", True)  # the p-adic code needs 64-bit integers
import galrax.padic as pf

## Zeros of a linear recurrence

Berstel's sequence $a_{n+3} = 2a_{n+2} - 4a_{n+1} + 4a_n$ from $0, 0, 1$: every $n \in \mathbb Z$ with $a_n = 0$.

In [3]:
pf.recurrence_zeros([2, -4, 4], [0, 0, 1])

ZeroSet(certified=True, zeros=(0, 1, 4, 6, 13, 52), period=None, by='real / real', primes=())

Every $n$ with $F_n = 2$ for Fibonacci, negative indices included:

In [4]:
pf.value_times([1, 1], [0, 1], 2).zeros

(-3, 3)

## Nonlinear maps

Every $n \ge 0$ with $x_n = 7$ along the orbit of $(1, 1)$ under $(x, y) \mapsto (y, xy + 1)$. Polynomials are `{exponents: coefficient}` (or Sage polynomials).

In [5]:
F = [{(0, 1): 1}, {(1, 1): 1, (0, 0): 1}]  # (y, x*y + 1)
zeros = pf.map_zeros(F, (1, 1), 7)
zeros

ZeroSet(certified=True, zeros=(4,), period=None, by='nilpotent', primes=(2,))

The certificate: one record per prime (or the real place), with the tail, the classes it covers and the type of argument.

In [6]:
zeros.records

({'p': 2,
  'k': 4,
  'type': 'nilpotent',
  'T': 14,
  'cover': [],
  'counted': [],
  'sides': 'forward'},)

## Will this loop ever stop?

`while x != 2*i + 22: i, x = i + 1, 2*x` from `i, x = 0, 1`, and the same loop with target `48` instead:

In [7]:
step = [{(1, 0): 1, (0, 0): 1}, {(0, 1): 2}]  # (i + 1, 2x)
print(pf.loop_exit(step, (0, 1), {(0, 1): 1, (1, 0): -2, (0, 0): -22}))  # 5 steps
print(pf.loop_exit(step, (0, 1), {(0, 1): 1, (0, 0): -48}))  # None: never

5


None


## Jump-ahead modulo $2^{64}$

The quadratic congruential generator $x \mapsto 2x^2 + 3x + 1$: its $10^{30}$-th iterate without iterating, and back from the value to the index.

In [8]:
qcg = pf.PAdicMap(2, [1, 3, 2])
y = qcg.iterate(0, 10**30, 64)
print(y, qcg.index(0, y, 64) == 10**30 % 2**64)

2322728876075122688 True


## An exact p-adic flow

$dx/dt = 3(x^3 - x)$ on $\mathbb Z_3$ to 19 digits: the flow at any $3$-adic time, exactly, and back.

In [9]:
import numpy as np


flow = pf.PAdicFlow(p=3, precision=19, field=[0, -3, 0, 3])
x = np.array([0, 1, 2, 5])
y = flow.flow(t=12345, x=x)
print(y, np.array_equal(flow.flow(-12345, y), x))

[        0         1  78198725 448129886] True


More in the [README](https://github.com/NotTruecCoder/galrax).